In [11]:
import math
import csv

In [12]:
# Сигмоид
f_sig = lambda x: 1/(1 + math.exp(-1*x))
# Конечно-разностный аналог сигмоида для метода обр распределения
anti_f_sig = lambda x: (1 - x)*x

# Тангенсоид
f_tang = lambda x:(math.exp(2*x) - 1) / (math.exp(2*x) + 1)
# Конечно-разностный аналог Тангенсоид для метода обр распределения
f_anti_tang = lambda x: 1 - x**2

In [13]:
class NeuralNetwork:
#     Необходимо помнить, что hiddenWeights должен быть передан
#     [
#         [w1, w4, w7]
#         [w2, w5, w8]
#         [w3, w6, w9]
#     ]
#     
    def __init__(self, activationFunc, deactivationFunc, hiddenWeights: list, outputWeights: list, requestedEra: int):
        self.requestedEra = requestedEra
        self.hiddenWeights = hiddenWeights   # Веса скрытых синапсов
        self.outputWeights = outputWeights     # Веса выходных синапсов
        self.activationFunc = activationFunc
        self.deactivationFunc = deactivationFunc
        
        
        self.totalMSE = 0.0
        
        
        self.E = 0.7     # Скорость обучения
        self.a = 0.3     # Момент
        
        self.dataset = []
        self.answersList = []
        
        self.hiddenWeightDeltas = []      # Дельты скрытых синапсов
        self.outputWeightDeltas = []      # Дельты выходных синапсов
        
    def readFromCsv(self, datasetPath: str):
        # Чтение датасета из csv
        with open(datasetPath, 'r') as csvDataset:
            csvDatasetReader = csv.reader(csvDataset, delimiter='\t')
            for row in csvDatasetReader:
                self.dataset.append([ float(row[0]), float(row[1]), float(row[2]) ])
    
    def train(self):
        for trainSet in self.dataset:
            print(trainSet)
            # Поиск идеального значения
            ideal = self.__calcDiscriminant(trainSet[0], trainSet[1], trainSet[2])

            for weight in self.hiddenWeights:
                print(f'weight: {weight}')
            hiddenInputs = self.__calculateHiddenInputs(trainSet)
            hiddenOutputs = self.__calculateHiddenOutputs(hiddenInputs)

            output = self.__calculateOutputs(hiddenOutputs)

            # Запись ответа для расчёта MSE
            self.answersList.append(output)
            self.totalMSE = self.__calculateMSE(ideal, output)

            hiddenDeltas, outputDeltas = self.__calculateDeltas(trainSet, output, ideal)
            
            self.hiddenWeights = self.__calculateNewHiddenWeigths(self.hiddenWeights, hiddenDeltas)
            self.outputWeights = self.__calculateNewOutputWeigths(self.outputWeights, outputDeltas)
            
    
    def __calcDiscriminant(self, a: float, b: float, c: float):
        return b**2 - 4*a*c
    
    def __calculateMSE(self, ideal:float, real: float):
        mse = float(0)
        for answer in self.answersList:
            mse += ((ideal - real)**2)
        
        return mse / len(self.answersList)
    
    # Приходит [I_1, I_2, I_3]
    # Возвращается [H1_in, H2_in, H3_in]
    def __calculateHiddenInputs(self, trainSet: list) -> list:
        result = []
        for weights in self.hiddenWeights:
            halfResult = float(0)
            for i in range(len(weights)):
                halfResult += trainSet[i]*weights[i]
            
            result.append(halfResult)
        
        return result
    
    # Приходит [H1_in, H2_in, H3_in]
    # Возвращается [H1_out, H2_out, H3_out]
    def __calculateHiddenOutputs(self, inputs: list) -> list:
        result = []
        
        print(inputs)
        for inputValue in inputs:
            result.append(self.activationFunc(inputValue))
        print()
        
        return result
    
    # Приходит [H1_out, H2_out, H3_out]
    # Возвращается O_out
    def __calculateOutputs(self, hiddenOutputs: list) -> float:
        result = float(0)
        for i in range(len(hiddenOutputs)):
            result += hiddenOutputs[i]*self.outputWeights[i]
        
        return result
    
    # trainSet: [a, b, c]
    # output: float
    def __calculateDeltas(self, trainSet: list, output: float, ideal: float):
        dO = (ideal - output)*self.deactivationFunc(output)
        
        dH = []
        for outputWeight in self.outputWeights:
            dH.append( (1 - output**2)*dO*outputWeight )
        
        # Расчёт дельт для весов скрытых синапсов
        hiddenGradients = []
        for dH_n in dH:
            buffer = []
            for inputValue in trainSet:
                buffer.append( inputValue*dH_n )
            
            hiddenGradients.append(buffer)
        
        hiddenDeltas = []
        for i in range(len(hiddenGradients)):
            deltasOfLine = []
            for j in range (len(hiddenGradients[i])):
                hiddenGradient = hiddenGradients[i][j]
                previousHiddenDelta = float(0)
                try:
                    previousHiddenDelta = float(self.hiddenWeightDeltas[i])
                except:
                    pass

                delta = self.E*hiddenGradient + self.a*previousHiddenDelta
            
                deltasOfLine.append(delta)
            
            hiddenDeltas.append(deltasOfLine)
        
        # Расчёт дельт для весов выходных синапсов
        outputDeltas = []
        for i in range(len(dH)):
            outputGradient = dO*dH[i]
            
            previousOutputDelta = float(0)
            try:
                previousOutputDelta = float(self.outputWeightDeltas[i])
            except:
                pass
            
            delta = self.E*outputGradient + self.a*previousOutputDelta
            
            outputDeltas.append(delta)
        
        return hiddenDeltas, outputDeltas
    
    
    def __calculateNewHiddenWeigths(self, currentWeights: list, deltas: list):
        newWeights = []
        # Рачёт новых весов для скрытых синапсов
        for i in range(len(currentWeights)):
            lineOfNewWeights = []
            for j in range(len(currentWeights[i])):
                lineOfNewWeights.append( currentWeights[i][j] + deltas[i][j] )
            
            newWeights.append(lineOfNewWeights)
            
        return newWeights
    
    def __calculateNewOutputWeigths(self, currentWeights: list, deltas: list):
        newWeights = []
        # Рачёт новых весов для выходных синапсов
        for i in range(len(currentWeights)):
            newWeights.append( currentWeights[i] + deltas[i] )
            
        return newWeights
            
    

In [14]:
hiddenWeights = [[0.1, 0.9, 0.31], [0.4, 0.7, 0.11], [0.3, 0.2, 0.27]]
outputWeights = [0.47, 0.51, 0.67]

In [15]:
myAI = NeuralNetwork(f_tang, f_anti_tang, hiddenWeights, outputWeights, 1000)

In [16]:
myAI.readFromCsv("test_data.csv")

In [17]:
myAI.train()

[3.0, 5.0, 2.0]
weight: [0.1, 0.9, 0.31]
weight: [0.4, 0.7, 0.11]
weight: [0.3, 0.2, 0.27]
[5.42, 4.92, 2.44]

[5.0, 6.0, 2.0]
weight: [-1.7015597085900331, -2.1025995143167218, -0.8910398057266888]
weight: [-1.554883939108334, -2.5581398985138906, -1.1932559594055558]
weight: [-2.2681808611815373, -4.0803014353025615, -1.4421205741210248]
[-22.905475240303872, -25.509771005436125, -38.7069540659651]

[11.0, 40.0, 77.0]
weight: [-0.6395414453759829, -0.8281775984598614, -0.46623250044106873]
weight: [-0.4024811428547903, -1.175256543009638, -0.7322948409041383]
weight: [-0.7542399327700187, -2.2635723212087395, -0.8365442027564174]
[-76.06196237149256, -107.82425704140687, -163.2534357210639]

[71.0, 69.0, 38.0]
weight: [434764.48313255137, 1580963.2542732987, 3043355.3924854766]
weight: [471766.0072289387, 1715513.0418710262, 3302364.13567573]
weight: [619770.8036144881, 2253712.4922619364, 4338400.068436744]
[255602247.76171687, 277355623.5580332, 364369091.6232985]


OverflowError: math range error

In [ ]:
print("myAI.totalMSE", myAI.totalMSE)
for weight in myAI.hiddenWeights:
    print(weight)
# print("myAI.hiddenWeights", myAI.hiddenWeights)
print("myAI.outputWeights", myAI.outputWeights)
print("myAI.answersList", myAI.answersList)